In [ ]:
import numpy as np
import pandas as pd
import gensim
from gensim.models import Word2Vec
from nltk.tokenize import word_tokenize
from sklearn.metrics.pairwise import cosine_similarity

# 1. Data Collection and Preprocessing (Using Twitter Dataset)
df = pd.read_csv('twitter_dataset.csv')
text_column = 'tweet' # Replace with your actual tweet column name

tweets = df[text_column].astype(str).tolist()

# Tokenize and lowercase tweets into lists of words
sentences = [word_tokenize(tweet.lower()) for tweet in tweets]

# 2. Train Word2Vec CBOW Model (sg=0) to learn word embeddings
model = Word2Vec(sentences=sentences, vector_size=50, window=5, min_count=1, sg=0, epochs=10)

# Document Representation: Calculate average word embedding for each tweet
def get_tweet_vector(words, model, vector_size=50):
    vectors = [model.wv[w] for w in words if w in model.wv]
    if len(vectors) == 0:
        return np.zeros(vector_size)
    return np.mean(vectors, axis=0)

tweet_vectors = np.array([get_tweet_vector(tweet, model, vector_size=50) for tweet in sentences])

# 3. Recommendation / Similar Tweet Retrieval Logic
def recommend_similar_tweets(user_query, tweet_vectors, tweets, model, top_n=3):
    # Preprocess and vectorize the user's input search/tweet
    query_words = word_tokenize(user_query.lower())
    query_vector = get_tweet_vector(query_words, model, vector_size=50).reshape(1, -1)
    
    # Calculate cosine similarity between query and all tweets in the dataset
    similarities = cosine_similarity(query_vector, tweet_vectors)[0]
    
    # Get top matching tweet indices
    top_indices = similarities.argsort()[::-1][:top_n]
    
    print(f"User Query / Input Tweet: '{user_query}'\n")
    print("Top Similar Tweets Recommended:")
    for idx in top_indices:
        print(f"  - [Similarity: {similarities[idx]:.4f}] {tweets[idx]}")

# Example Usage: Find tweets similar to a user search or input text
recommend_similar_tweets("excited for the weekend plans", tweet_vectors, tweets, model, top_n=3)